# Unrolling: two compilations, two meanings

There are two honest things a dynamic system can mean, and conflating them is how dynamic
models get fit wrong.

* **Conditional form** — `y = f(x, y.l1, ...)` with `y.l1` a real column of the panel. One
  row per unit-period, the ordinary dynamic regression. Only meaningful when every lagged
  endogenous variable it reads is *measured*.
* **Marginal form** — `y.t3 = f(x.t0, ..., x.t3)`, the system substituted into itself period
  by period. This is what a latent state needs, what a policy simulation evaluates, and what
  the identification machinery reads.

In [ ]:
import numpy as np
import pandas as pd

from axiom.core import D, Likelihood, Param, Prior, dimensionless, latex, log_likelihood, value
from axiom.dynamics import (
    Form, Unrolled, Variable, conditional_form, lagged_columns, parse_system, prepare_panel,
    to_model_spec, unroll, unrolled_edges,
)

from axiom.display import enable, show_math, table

enable();  # every axiom result renders itself from here on

NONE = dimensionless()
system = parse_system(
    "stock = decay * stock[t-1] + beta * inflow",
    variables=(
        Variable(name="stock", dimension=D.outcome),
        Variable(name="inflow", dimension=D.currency, role="exogenous"),
    ),
    parameters=(
        Param(name="decay", dimension=NONE),
        Param(name="beta", dimension=D.outcome / D.currency),
    ),
    name="one-compartment",
)

## The marginal form

`unroll` substitutes the system into itself. `stock.t2` ends up written in terms of
`inflow.t0`, `inflow.t1`, `inflow.t2` and nothing else — every path from an input to the
outcome is explicit, which is what the identifiability analysis needs.

In [ ]:
marginal = unroll(system, periods=4)
form: Form = marginal.form
print("form:", form, "| nodes:", marginal.node_count, "| exact:", marginal.exact)
print("columns:", marginal.columns)
show_math(marginal.expression("stock", 2))

In [ ]:
# It is an ordinary Expr, so `value` evaluates it — and it matches the loop it compiles.
inflow = [1.0, 2.0, 0.5, 3.0]
data = {f"inflow.t{t}": np.array(v) for t, v in enumerate(inflow)}
theta = {"decay": 0.6, "beta": 2.0}
compiled = [float(np.ravel(value(marginal.expression("stock", t), data=data, params=theta))[0]) for t in range(4)]
loop, running = [], 0.0
for v in inflow:
    running = 0.6 * running + 2.0 * v
    loop.append(running)
print(np.round(compiled, 8), np.round(loop, 8), np.allclose(compiled, loop))

## The conditional form and its panel

`conditional_form` writes one generic period. The lags it needs become real columns;
`prepare_panel` builds them per unit, in time order, filling a lag that reaches before a
unit's first period with the variable's declared `initial`.

In [ ]:
conditional = conditional_form(system)
print("columns:", conditional.columns, "| lagged:", lagged_columns(system))
show_math(conditional.expression("stock"))

frame = pd.DataFrame({
    "unit": ["a"] * 4 + ["b"] * 4,
    "t": list(range(4)) * 2,
    "inflow": [1.0, 2.0, 0.5, 3.0, 0.0, 1.0, 1.0, 1.0],
    "stock": loop + [0.0, 2.0, 3.2, 3.92],
})
prepared = prepare_panel(system, frame, unit="unit", time="t")
print(prepared.head(6).to_string(index=False))

### A latent lag has no column, and the compiler says so

Substituting an unobserved lag with data you do not have is the bug this check exists to
prevent. The marginal form is the route that does not need it.

In [ ]:
latent = parse_system(
    "state = decay * state[t-1] + shock",
    variables=(
        Variable(name="state", dimension=NONE, observed=False),
        Variable(name="shock", dimension=NONE, role="exogenous"),
    ),
    parameters=(Param(name="decay", dimension=NONE),),
)
refused = conditional_form(latent)
print(type(refused).__name__, "| missing:", refused.missing)
print(refused.reason)

## From a compiled node to a fitted model

`to_model_spec` wraps one compiled expression as a `ModelSpec`, which any backend fits.
Nothing about it is special: it is the same spec type a hand-written model produces.

In [ ]:
model = to_model_spec(
    conditional,
    "stock",
    likelihood=Likelihood(family="normal", scale="sigma"),
    priors={
        "decay": Prior(family="beta", hyper={"alpha": 1.5, "beta": 1.5}),
        "beta": Prior(family="lognormal", hyper={"mu": 0.0, "sigma": 1.0}),
    },
    extra_parameters=(
        Param(name="sigma", dimension=D.outcome, prior=Prior(family="halfnormal", hyper={"sigma": 1.0})),
    ),
)
print(model.name, "|", [p.name for p in model.parameters], "| reads:", model.data_columns)
fit_data = {name: prepared[name].to_numpy() for name in ("stock", "stock.l1", "inflow")}
print("log likelihood:", round(log_likelihood(model, fit_data, {"decay": 0.6, "beta": 2.0, "sigma": 0.5}), 4))

## The unrolled graph

`unrolled_edges` gives the time-indexed dependency edges. The default is the *reduced* form,
which is acyclic even for a simultaneous system: the members of a block share the block's
parents and have no arrows between them. That is the claim that no ordering of them is
causal within the period.

In [ ]:
table([[str(edge)] for edge in unrolled_edges(system, periods=3)], headers=("unrolled edge",))
print("as written (may be cyclic):", unrolled_edges(system, periods=2, reduced=False))

## Seeing it

`enable()` at the top of this notebook already made a bare result on the last
line of a cell render itself — a card drawn by `rich`, or the same content as
aligned plain text where `rich` is not installed. `show` does it on demand, for
a result that is not the last thing in its cell.

`axiom.viz` draws the figure this subpackage's results are actually about.

In [ ]:
from axiom.core import D, Param, dimensionless
from axiom.display import show
from axiom.dynamics import Variable, parse_system, unroll
from axiom.viz import unrolled

system = parse_system(
    "stock = decay * stock[t-1] + beta * inflow",
    variables=(
        Variable(name="stock", dimension=D.outcome, initial=0.0),
        Variable(name="inflow", dimension=D.currency, role="exogenous"),
    ),
    parameters=(
        Param(name="decay", dimension=dimensionless()),
        Param(name="beta", dimension=D.outcome / D.currency),
    ),
    name="one-compartment",
)
result = unroll(system, periods=4)
show(result)
unrolled(result)

Feedback becomes a DAG once time is explicit, and here the horizontal axis *is* the period.